# 28-Day Future Predictions

Loads all 4 tuned 28-day models and runs them on the future feature CSV.
Saves one predictions CSV alongside the future data.

| | |
|---|---|
| **Input**  | `data/future/28_days_window/future_2016_04_30.csv` |
| **Models** | `artifacts/models/28_day_models/*.joblib` |
| **Output** | `data/future/28_days_window/predictions_2016_04_30.csv` |

**Forecast window:** Apr 30 – May 27, 2016 (1,437 SKUs)

## 1) Imports & Paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

ROOT       = Path('..').resolve()
FUTURE_DIR = ROOT / 'data' / 'future' / '28_days_window'
MODELS_DIR = ROOT / 'artifacts' / 'models' / '28_day_models'

FUTURE_CSV = FUTURE_DIR / 'future_2016_04_30.csv'
OUT_CSV    = FUTURE_DIR / 'predictions_2016_04_30.csv'

print(f'Future CSV : {FUTURE_CSV}')
print(f'Models dir : {MODELS_DIR}')
print(f'Output     : {OUT_CSV}')

Future CSV : C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\future\28_days_window\future_2016_04_30.csv
Models dir : C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\artifacts\models\28_day_models
Output     : C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\future\28_days_window\predictions_2016_04_30.csv


## 2) Load Future Feature CSV

In [2]:
future_df = pd.read_csv(FUTURE_CSV, parse_dates=['date'])

DROP     = ['item_id', 'date']
FEATURES = [c for c in future_df.columns if c not in DROP]

X_future = future_df[FEATURES]

print(f'SKUs     : {len(future_df)}')
print(f'Date     : {future_df["date"].iloc[0].date()}')
print(f'Features : {len(FEATURES)}')
print(f'\n{future_df.head(3).to_string(index=False)}')

SKUs     : 1437
Date     : 2016-04-30
Features : 31

    item_id       date  is_month_end  aggregated_sell_price  snap_ca  event_christmas_28  event_easter_28  event_eid_al_fitr_28  event_eid_al_adha_28  event_fathers_day_28  event_halloween_28  event_mothers_day_28  event_newyear_28  event_orthodox_christmas_28  event_orthodox_easter_28  event_ramadan_starts_28  event_thanksgiving_28  event_valentines_day_28  event_superbowl_28  event_independence_day_28  event_memorial_day_28  event_labor_day_28  event_mlk_day_28  event_presidents_day_28  event_columbus_day_28  event_veterans_day_28  event_st_patricks_day_28  event_cinco_de_mayo_28  event_chanukah_28  event_lent_start_28  event_lent_week2_28  event_pesach_end_28  event_purim_end_28
FOODS_1_001 2016-04-30             1                   2.24        1                   0                0                     0                     0                     0                   0                     1                 0                         

## 3) Load All 4 Tuned Models

In [3]:
MODEL_FILES = {
    'pred_lightgbm':       'lightgbm_28d_tuned.joblib',
    'pred_xgboost':        'xgboost_28d_tuned.joblib',
    'pred_random_forest':  'random_forest_28d_tuned.joblib',
    'pred_neural_network': 'neural_network_28d_tuned.joblib',
}

models = {}
for col, fname in MODEL_FILES.items():
    models[col] = joblib.load(MODELS_DIR / fname)
    print(f'Loaded: {fname}')

Loaded: lightgbm_28d_tuned.joblib
Loaded: xgboost_28d_tuned.joblib
Loaded: random_forest_28d_tuned.joblib
Loaded: neural_network_28d_tuned.joblib


## 4) Run Predictions

In [4]:
pred_df = future_df[['item_id', 'date']].copy()

for col, model in models.items():
    raw = model.predict(X_future)
    pred_df[col] = np.ceil(np.clip(raw, 0, None)).astype(int)
    print(f'{col}: mean={pred_df[col].mean():.2f}  min={pred_df[col].min()}  max={pred_df[col].max()}')

print(f'\nPredictions shape: {pred_df.shape}')
print(pred_df.head(10).to_string(index=False))

pred_lightgbm: mean=50.94  min=13  max=266
pred_xgboost: mean=53.22  min=17  max=247
pred_random_forest: mean=51.07  min=4  max=323
pred_neural_network: mean=51.41  min=13  max=203

Predictions shape: (1437, 6)
    item_id       date  pred_lightgbm  pred_xgboost  pred_random_forest  pred_neural_network
FOODS_1_001 2016-04-30             32            35                  30                   41
FOODS_1_002 2016-04-30             18            23                  13                   13
FOODS_1_003 2016-04-30             51            48                  49                   39
FOODS_1_004 2016-04-30             43            51                  46                   60
FOODS_1_005 2016-04-30             27            47                  25                   39
FOODS_1_006 2016-04-30             46            42                  39                   36
FOODS_1_008 2016-04-30             27            47                  25                   39
FOODS_1_009 2016-04-30             32        

c:\Users\amrok\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\amrok\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\amrok\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\amrok\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\e

## 5) Save Predictions CSV

In [5]:
pred_df.to_csv(OUT_CSV, index=False)

print(f'Saved: {OUT_CSV}')
print(f'Rows : {len(pred_df):,}')
print(f'Cols : {list(pred_df.columns)}')

Saved: C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\future\28_days_window\predictions_2016_04_30.csv
Rows : 1,437
Cols : ['item_id', 'date', 'pred_lightgbm', 'pred_xgboost', 'pred_random_forest', 'pred_neural_network']
